In [3]:
import struct
import zlib
import json
import lzma
import math

filename = "Zeldas Adventure (Europe).chd"
#filename = "Link - The Faces of Evil (Europe).chd"
#filename = "Zelda - The Wand of Gamelon (USA).chd"

# Subcode seems to be stripped from the CD-I files I have access to.
IGNORE_SUBCODE = True

with open(filename, "rb") as f:
    gameFile = f.read()
    
len(gameFile), gameFile[:8]

(185978385, b'MComprHD')

# CHD spec resources

Headers: https://github.com/mamedev/mame/blob/master/src/lib/util/chd.h

# Reverse-Engineered Spec Parts

## Metadata Entries

Metadata is stored as a linked list of data blobs. Each blob has a tag that indicates how
the data should be interpreted. The info has been formatted in the style of chd.h. The data
blob immediately follows the end of the header.

```
[ 0] char tag[4];         // Tag used to interpret data blob 
[ 4] uint8_t flags;       // Byte with flags related to the categories that the tag is in
[ 5] UINT24 dataLength;   // The length of the data blob in bytes
[ 8] uint64_t nextOffset; // The offset in the file of the next metadata entry
[16] (metadata entry length)
```

## Compressed Map Format

The compressed map data uses a custom compression format that immediately follows the header.
The format begins with a run-length encoded huffman tree of nibbles (4-bit values). Then an array of
compression types is run-length encoded, and then huffman encoded using the above tree. Finally, there
is an array of non-standard-bit-length numbers, using the compression type array and the bit lengths
specified in the compression header.

### Huffman Tree

The data immediately following the header is run-length encoded, and must be decoded until 16 nibbles
have been read. In the run-length encoding, any number other than 1 is copied to the output. When 1 is
read, if followed by another 1, then a "1" is output. Otherwise, the 1 is followed by the number to
repeat, and then a nibble N, where N + 3 is the number of times it should be repeated.

For example, the hex data: `0x21156688361836`
Becomes the nibble array: `2, 1, 5, 6; 6, 8, 8, 3; 6, 8, 8, 8; 8, 8, 8, 6`

The output nibbles are interpreted as the bitlengths of their index's huffman code word. That is
enough information to reconstruct the tree. Code words with the same length are assigned in increasing
order, and shorter paths have more "1" bits than longer paths.

For example, given the nibble array: `2, 1, 5, 6; 6, 8, 8, 3; 6, 8, 8, 8; 8, 8, 8, 6`

The nibbles describe their index's code word, so the above array in words is:

- 1 is represented with 1 bit
- 0 is represented with 2 bits
- 7 is represented with 3 bits
- 2 is represented with 5 bits
- 3, 4, 8, and F are each represented with 6 bits
- 5, 6, 9, A, B, C, D, and E are each represented with 8 bits

Which describes this tree:
```
1b = 0x1
01b = 0x0
001b = 0x7
00011b = 0x2
000101b = 0xF
000100b = 0x8
000011b = 0x4
000010b = 0x3
00000111b = 0xE
00000110b = 0xD
00000101b = 0xC
00000100b = 0xB
00000011b = 0xA
00000010b = 0x9
00000001b = 0x6
00000000b = 0x5
```


<aside style="background-color:whitesmoke;padding:2em;">
<h4>Commentary</h4>
<p>
The choice of 1 for the run length encoding's command value is ok but not good. There is a guarantee
that, if a 1 is present, it will either appear exactly once, or it will appear twice and all other
values will be 0. It's fairly common for a 1 to be present in a huffman tree.
</p>
<p>
An optimal choice for the command value is -1 (or 2<sup>maxbits</sup> - 1). With that value, there are exactly
4 input strings which need to be escaped, because exactly one other nibble in the data stream has a
value that is not -1. Those input strings are (in the 16 bit case): 0xF1FFFF..., 0xFF1FFFF...,
0xFFF...F1FF, and 0xFFF...F1F. That ambiguity can be resolved by changing the +3 in the repeat command
to +1 when the value to be repeated is (2<sup>maxbits</sup> - 1). In practice, these huffman trees never appear,
so the cost of escaping will never be paid.
</p>
</aside>

### Compression Type Array

The compression type array follows the huffman tree. This data is huffman encoded, but it has an unknown length
and must be decoded as a stream. For each decoded nibble, if the value is less than 7, add the nibble to the
output array. If the value is 7, read another nibble N, and repeat the previously output value N + 2 times. If
the value is 8, read a byte B (two nibbles in big-endian order), and repeat the previously output value B + 18 times.

### Compressor Data

Most of the compression types in the compression type array need extra info about the hunk of data. This extra
data follows the compression type array. For compression types 0, 1, 2, and 3, the length of the compressed hunk
is stored in the next `lengthbits` bits. For compression NONE, there is no extra data. For compression SELF, the
file offset of another hunk is stored in the next `selfbits` bits. Similarly for PARENT and `parentbits`.

Map compressor enum:
TYPE_0 = 0 // Index 0 in the file header's compressors array.
TYPE_1 = 1 // Index 1 in the file header's compressors array.
TYPE_2 = 2 // Index 2 in the file header's compressors array.
TYPE_3 = 3 // Index 3 in the file header's compressors array.
NONE = 4
SELF = 5
PARENT = 6



In [4]:
# Note: This stream is big-endian. The highest bit in a byte has index 0. All operations have a maximum of 24 bits.
# Based on https://github.com/mamedev/mame/blob/ee1e4f9683a4953cb9d88f9256017fcbc38e3144/src/lib/util/bitstream.h
# but fixes a few bugs
class BitStream:
    def __init__(self, byteArray, cursor = 0):
        assert cursor >= 0
        self.bytes = byteArray
        self.cursor = 0
        self._byteCursor = 0
        self._accumulator = 0
        self._bitsInAccumulator = 0
        
        if cursor != 0:
            self.seek(cursor)
    
    def __len__(self):
        return len(self.bytes) * 8
    
    def __repr__(self):
        return "BitStream(" + repr(self.bytes) + ", cursor = " + str(self.cursor) +")"
    
    def seek(self, offset):
        assert offset >= 0
        assert offset < len(self)
        
        self._byteCursor = offset // 8
        offsetBits = offset % 8
        self._accumulator = 0
        self._bitsInAccumulator = 0
        
        self.cursor = offset - offsetBits
        self.remove(offsetBits)
    
    def _fetch(self, count):
        while self._bitsInAccumulator < count:
            if self._byteCursor < len(self.bytes):
                self._accumulator |= self.bytes[self._byteCursor] << (24 - self._bitsInAccumulator)
            self._byteCursor += 1
            self._bitsInAccumulator += 8
    
    def peek(self, count):
        assert count >= 0
        assert count <= 24
        if count == 0:
            return 0
        
        self._fetch(count)
        
        return self._accumulator >> (32 - count)
    
    def remove(self, count):
        assert count >= 0
        if count == 0:
            return
        assert count <= 24
        
        self._fetch(count)
        self._removeWithoutFetch(count)
    
    def _removeWithoutFetch(self, count):
        assert count <= self._bitsInAccumulator
        self._accumulator = (self._accumulator << count) & 0xFFFF_FFFF
        self._bitsInAccumulator -= count
        self.cursor += count
        
    def read(self, count):
        assert count >= 0
        assert count <= 24
        
        ret = self.peek(count)
        self._removeWithoutFetch(count)
        
        return ret
            
test = BitStream(b'12345', cursor = 1)
value = test.peek(24)
expectedValue = ((b'1'[0] << 16) + (b'2'[0] << 8) + b'3'[0]) << 1
assert value == expectedValue, value

test = BitStream(bytearray([0x21, 0x56, 0x68, 0x83, 0x68, 0x88, 0x88, 0x86]))
value = test.read(4)
assert value == 2, value
value = test.read(4)
assert value == 1, value
value = test.read(4)
assert value == 5, value

del test
del value
del expectedValue

print("All tests passed")

All tests passed


In [5]:
# Based heavily on https://github.com/mamedev/mame/blob/ee1e4f9683a4953cb9d88f9256017fcbc38e3144/src/lib/util/huffman.cpp
class HuffmanCodedData:
    def __init__(self, bitbuf):
        self.maxbits = 8
        self.numcodes = 16
        
        self.huffnodes = [HuffmanCodedDataNode(i) for i in range(self.numcodes)]
        self.lookup = [None] * (1 << self.maxbits)
        
        self._readTree(bitbuf)
        
    
    def _readTree(self, bitbuf):
        numbits = 4
        
        curnode = 0
        while curnode < self.numcodes:
            nodebits = bitbuf.read(numbits)
            if nodebits != 1:
                self.huffnodes[curnode].numbits = nodebits
                curnode += 1
            else:
                nodebits = bitbuf.read(numbits)
                if nodebits == 1:
                    self.huffnodes[curnode].numbits = 1
                    curnode += 1
                else:
                    repcount = bitbuf.read(numbits) + 3
                    for i in range(repcount):
                        self.huffnodes[curnode + i].numbits = nodebits
                    curnode += repcount
        assert curnode == self.numcodes
        
        self._assignCanonicalCodes()
        self._buildLookupTable()
        
    def _assignCanonicalCodes(self):
        histogram = [0] * 33
        for node in self.huffnodes:
            histogram[node.numbits] += 1
        
        codeStartNumbers = [0] * 33
        currentStart = 0
        for i in range(32, 0, -1):
            nextStart = (currentStart + histogram[i]) >> 1
            codeStartNumbers[i] = currentStart
            currentStart = nextStart
        
        for node in self.huffnodes:
            node.bits = codeStartNumbers[node.numbits]
            codeStartNumbers[node.numbits] += 1
    
    def _buildLookupTable(self):
        for node in self.huffnodes:
            if node.numbits > 0:
                shift = self.maxbits - node.numbits
                for i in range(node.bits << shift, (node.bits + 1) << shift):
                    self.lookup[i] = node
    
    def readOne(self, bitbuf):
        node = self.lookup[bitbuf.peek(self.maxbits)]
        bitbuf.remove(node.numbits)
        return node.value
        
                
class HuffmanCodedDataNode:
    def __init__(self, value, numbits = 0, bits = 0):
        self.numbits = numbits
        self.bits = bits
        self.value = value
    
    def codedValue(self):
        return ("{:0" + str(self.numbits) + "b}").format(self.bits)
    
    def __repr__(self):
        return "HuffmanCodedDataNode({:04b}, numbits = {}, bits = {:b}, codedValue = {})" \
            .format(self.value, self.numbits, self.bits, self.codedValue())

test = HuffmanCodedData(BitStream(bytearray([0x21, 0x15, 0x66, 0x88, 0x36, 0x18, 0x36])))
assert [(n.numbits, n.bits) for n in test.huffnodes] == [
    (2, 1),
    (1, 1),
    (5, 3),
    (6, 2),
    (6, 3),
    (8, 0),
    (8, 1),
    (3, 1),
    (6, 4),
    (8, 2),
    (8, 3),
    (8, 4),
    (8, 5),
    (8, 6),
    (8, 7),
    (6, 5)
], print(str(test.huffnodes).replace("),", "),\n"))

assert None not in test.lookup
del test
print("All tests passed")

All tests passed


In [6]:
# Misc CD format info
# From https://github.com/mamedev/mame/blob/f1f77b1a1c4d99d78d0715a1e36ec9ab8e2c8f7d/src/lib/util/cdrom.h#L28
CDRomConstants = {
    # Tracks are padded to this many frames
    "TRACK_PADDING": 4,
    "MAX_SECTOR_DATA": 2352,
    "MAX_SUBCODE_DATA": 96,
    # CHD_FRAME should not be confused with CD frames!
    "CHD_FRAME_SIZE": 2352 + 96,
    "ECC_OFFSET": 0x81C,
    "ECC_LENGTH": 86 * 2 + 52 * 2
}

CD_SYNC_HEADER = bytes([
    0x00, 0xFF, 0xFF, 0xFF, 0xFF, 0xFF,
    0xFF, 0xFF, 0xFF, 0xFF, 0xFF, 0x00
])
CD_SYNC_HEADER

b'\x00\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\x00'

In [7]:


class ChdHunkMap:
    def __init__(self, data, hunkCount):
        mapDataLen = struct.unpack(">I", data[:4])[0]
        self.dataStartOffset = struct.unpack(">Q", b'\0\0' + data[4:10])[0]
        self.crc, self.lengthBits, self.hunkRefBits, self.parentRefBits = \
            struct.unpack(">H3B", data[10:15])
        bitBuffer = BitStream(data[16:mapDataLen])
        compressedData = HuffmanCodedData(bitBuffer)
        
        compressionTypeArray = []
        prevType = 0 # Arrays CAN start with a run length entry "repeating" this type before the first entry is added
                     # to the array!!
        while len(compressionTypeArray) < hunkCount:
            compressionType = compressedData.readOne(bitBuffer)
            if compressionType == 7:
                count = 3 + compressedData.readOne(bitBuffer)
                for _ in range(count):
                    compressionTypeArray.append(prevType)
            elif compressionType == 8:
                highPart = compressedData.readOne(bitBuffer)
                lowPart = compressedData.readOne(bitBuffer)
                count = 19 + (highPart << 4) + lowPart
                for _ in range(count):
                    compressionTypeArray.append(prevType)
            else:
                assert compressionType < 7, str(compressionType)
                compressionTypeArray.append(compressionType)
                prevType = compressionType
            
        assert len(compressionTypeArray) == hunkCount
        
        self.hunks = []
        currentOffset = self.dataStartOffset
        for compressor in compressionTypeArray:
            hunk = ChdHunk(compressor, currentOffset)
            if compressor < 4:
                hunk.compressedLength = bitBuffer.read(self.lengthBits)
                currentOffset += hunk.compressedLength
                hunk.crc = bitBuffer.read(16)
            else:
                raise Exception("Not implemented")
            self.hunks.append(hunk)
        
        print("Hunk end offset:", self.hunks[-1].offset)
        print("hunk count", len(self.hunks))
    
    def toDict(self):
        return {
            "crc": self.crc,
            "dataStartOffset": self.dataStartOffset,
            "lengthBits": self.lengthBits,
            "hunkRefBits": self.hunkRefBits,
            "parentRefBits": self.parentRefBits,
            "hunks": [h.toDict() for h in self.hunks]
        }
        

# Hunks are multiple sectors appended to each other.
class ChdHunk:
    def __init__(self, compressor, offset):
        self.compressorId = compressor
        self.compressorName = None
        self.compressedData = None
        self.data = None
        self.offset = offset
        self.compressedLength = None
        self.crc = None
        self.sectors = None
    
    def _dictSizeFromHunkLen(self, hunkLength):
        # Find the smallest power of 2 greater than or equal to 2^11 that can fit hunkLength
        for i in range(11, 32):
            if hunkLength < (1 << i):
                return 1 << i
        raise Exception()
    
    # Based on https://github.com/mamedev/mame/blob/master/src/lib/util/chdcodec.cpp#L382
    def decompress(self, data, hunkLength):
        self.compressedData = data
        assert hunkLength % CDRomConstants["CHD_FRAME_SIZE"] == 0
        
        # Compute header size
        chdFrames = hunkLength // CDRomConstants["CHD_FRAME_SIZE"]
        if hunkLength < 65536:
            complenBytes = 2
        else:
            complenBytes = 3
        eccBytes = (chdFrames + 7) // 8
        headerBytes = eccBytes + complenBytes
        
        
        # "Extract compressed length of base"... unsure what that means?
        try:
            complenBase = (data[eccBytes] << 8) + data[eccBytes + 1]
            if complenBytes > 2:
                complenBase = (complenBase << 8) + data[eccBytes + 2]
        except IndexError as e:
            #raise Exception("Hunk with offset {} doesn't have enough bytes; need at least {} bytes, but has {}." \
            #               .format(self.offset, headerBytes, len(data)))
            return
        
        assert headerBytes + complenBase <= len(data)
        baseData = data[headerBytes:headerBytes + complenBase]
        subcodeData = data[headerBytes + complenBase:]
        
        expectedLen = chdFrames * CDRomConstants["MAX_SECTOR_DATA"]
        if self.compressorName == 'cdzl':
            inflatedBaseData = zlib.decompress(baseData, wbits=-15)
            
            if len(inflatedBaseData) != expectedLen:
                print("Expected base data size", expectedLen, "found", len(inflatedBaseData), \
                      "with compressor", self.compressorName)
            
        elif self.compressorName == 'cdlz':
            decompressor = lzma.LZMADecompressor(format=lzma.FORMAT_RAW, filters=[
                {"id": lzma.FILTER_LZMA1, "preset": 9, "dict_size": self._dictSizeFromHunkLen(hunkLength)}
            ])
            inflatedBaseData = decompressor.decompress(baseData, max_length = hunkLength)
            if len(decompressor.unused_data) > 0:
                print("needs_input:", decompressor.needs_input, "unused length:", len(decompressor.unused_data))
                
            if len(inflatedBaseData) > expectedLen:
                excessData = inflatedBaseData[expectedLen:]
                if len(list(filter(lambda b: b != 0, excessData))):
                    print("Expected base data size", expectedLen, "found", len(inflatedBaseData), \
                      "with compressor", self.compressorName)
                    print("\tExtra data:", excessData)
                    
            while len(inflatedBaseData) < expectedLen:
                inflatedBaseData += b'\0'
        else:
            raise Exception("Not implemented: " + self.compressorName)

        if not IGNORE_SUBCODE:
            inflatedSubcodeData = zlib.decompress(subcodeData, wbits=-15)
            expectedLen = chdFrames * CDRomConstants["MAX_SUBCODE_DATA"]
            if len(inflatedSubcodeData) != expectedLen:
                print("Expected subcode data size", expectedLen, "found", len(inflatedSubcodeData))
        
        
        self.sectors = []
        eccBitvec = data[:eccBytes]
        for frameNumber in range(chdFrames):
            sectorStart = frameNumber * CDRomConstants["MAX_SECTOR_DATA"]
            subcodeStart = frameNumber * CDRomConstants["MAX_SUBCODE_DATA"]
            
            sectorData = inflatedBaseData[sectorStart:sectorStart + CDRomConstants["MAX_SECTOR_DATA"]]
            if not IGNORE_SUBCODE:
                subcodeData = inflatedSubcodeData[subcodeStart:subcodeStart + CDRomConstants["MAX_SUBCODE_DATA"]]
            
            # Error correcting code related stuff. CD Rom data structure will handle this.
            hasEcc = eccBitvec[frameNumber // 8] & (1 << (frameNumber % 8)) != 0
            
            self.sectors.append({
                "sector": sectorData,
                "hasEcc": hasEcc
            })
            if not IGNORE_SUBCODE:
                self.sectors[-1]["subcode"] = subcodeData
        
    
    def toDict(self):
        return {
            #"compressorId": self.compressorId,
            "compressorName": self.compressorName,
            "offset": self.offset,
            "sectors": self.sectors,
            "crc": self.crc
        }
    

class ChdMetadataEntry:
    def __init__(self, tag, flagByte, dataString):
        self.tag = tag
        self.flagByte = int(flagByte)
        self.dataString = dataString
        
        if tag == "CHT2":
            self.data = {}
            for kvpair in dataString.split():
                [key, value] = kvpair.split(':')
                self.data[key] = value
        else:
            self.data = None
    
    def to_dict(self):
        return {
            "tag": self.tag,
            "flagByte": self.flagByte,
            "dataString": self.dataString,
            "data": self.data
        }

class ChdFile:
    def __init__(self, data):
        # Main Header
        assert data[:8] == b'MComprHD'
        headerLen, versionNumber = struct.unpack(">II", data[8:16])
        assert versionNumber == 5
        assert headerLen == 124
        
        self.compressorCodecs = list(map(lambda b: b.decode('ascii'), filter(lambda b: b[0] != 0, \
                                            struct.unpack(">4s4s4s4s", data[16:32]))))
        self.uncompressedSize, mapOffset, metaOffset, self.hunkSize, self.unitSize = \
            struct.unpack(">QQQII", data[32:64])
        self.hunkCount = (self.uncompressedSize + self.hunkSize - 1) // self.hunkSize
        self.unitCount = (self.uncompressedSize + self.unitSize - 1) // self.unitSize
        self.rawSha1 = data[64:84]
        self.fullSha1 = data[84:104]
        self.parentSha1 = data[104:124]

        # Hunk Map
        print(hex(mapOffset))
        mapData = data[mapOffset:]
        self.map = ChdHunkMap(mapData, self.hunkCount)
        for hunk in self.map.hunks:
            if hunk.compressorId < 4:
                hunk.compressorName = self.compressorCodecs[hunk.compressorId]
            hunk.decompress(data[hunk.offset:hunk.offset + hunk.compressedLength], self.hunkSize)
        
        # Metadata
        self.metadata = []
        nextOffset = metaOffset
        while nextOffset != 0:
            rawTag, flags, lengthBytes, nextOffset = \
                struct.unpack(">4sB3sQ", data[metaOffset:metaOffset + 16])
            length = struct.unpack(">I", b'\0' + lengthBytes)[0]
            rawMetadataString = data[metaOffset + 16:metaOffset + 16 + length]
            metadataString = rawMetadataString.split(b'\0')[0].decode('ascii')
            tag = rawTag.split(b'\0')[0].decode('ascii')
            self.metadata.append(ChdMetadataEntry(tag, flags, metadataString))
            
    def toDict(self):
        return {
            "compressorCodecs": self.compressorCodecs,
            "uncompressedSize": self.uncompressedSize,
            "hunkSize": self.hunkSize,
            "unitSize": self.unitSize,
            "rawSha1": self.rawSha1,
            "fullSha1": self.fullSha1,
            "parentSha1": self.parentSha1,
            "map": self.map.toDict(),
            "metadata": list(map(lambda m: m.to_dict(), self.metadata))
        }

gameChd = ChdFile(gameFile)
len(gameChd.map.hunks)

0xb1427c7
Hunk end offset: 185862499
hunk count 27507


27507

In [8]:
gameChd.__dict__

{'compressorCodecs': ['cdlz', 'cdzl', 'cdfl'],
 'uncompressedSize': 538687296,
 'hunkSize': 19584,
 'unitSize': 2448,
 'hunkCount': 27507,
 'unitCount': 220052,
 'rawSha1': b'G\xd2.Z`\x8b\x7f\x05\x82\xeb\xd6\xa8\xa3\x029\xdaH\x9a\xee\x84',
 'fullSha1': b"\x1f\xd5\x81\\V\xfbynX'Q,(\xe3\xd8\r[\xdfYj",
 'parentSha1': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
 'map': <__main__.ChdHunkMap at 0x125bf90ec50>,
 'metadata': [<__main__.ChdMetadataEntry at 0x125b46bb208>]}

In [155]:
def fromBcd(n):
    return (n // 16 * 10) + (n % 16)
    

class CDImage:
    def __init__(self, chdFile):
        self.allSectors = []
        
        # The first two seconds of sectors (ToC) are not present in chd format.
        index = 75 * 2
        for hunk in chdFile.map.hunks:
            if hunk.sectors:
                for chdSector in hunk.sectors:
                    cdSector = CDSector(chdSector, index)
                    self.allSectors.append(cdSector)
                    index += 1

class CDSector:
    def __init__(self, chdSector, index):
        self._debug_chdSector = chdSector
        
        if not IGNORE_SUBCODE:
            self.rawSubcode = chdSector["subcode"]
        rawSector = chdSector["sector"]
        if not chdSector["hasEcc"] and rawSector[:12] != CD_SYNC_HEADER:
            # TODO: Read subcode channel Q?
            self.kind = "rawAudio"
            self.data = rawSector
            self.sectors = index % 75
            self.seconds = (index // 75) % 60
            self.minutes = index // (75 * 60)
            
            
            
            # Unused fields
            self.mode = None
            self.crc32 = None
            self.ecc = None
        
        else:
            self.kind = "data"
            # First 12 bytes are sync pattern, ignore them.
            # Next three bytes are sector address
            self.minutes = fromBcd(rawSector[12])
            self.seconds = fromBcd(rawSector[13])
            self.sectors = fromBcd(rawSector[14])
            self.mode = rawSector[15]
            
            assert self.sectors == index % 75, (self.sectors, index, index % 75)
            assert self.seconds == (index // 75) % 60, (self.seconds, index, (index // 75) % 60)
            assert self.minutes == index // (75 * 60), (self.minutes, index, index // (75 * 60))
            #if self.sectors != index % 75:
            #    print(self.sectors, index, index % 75)
            #if self.seconds != (index // 75) % 60:
            #    print(self.seconds, index, (index // 75) % 60)
            #if self.minutes != index // (75 * 60):
            #    print(self.minutes, index, index // (75 * 60))
            
            if self.mode == 1:
                self.data = rawSector[16:16 + 2048]
                self.crc32 = rawSector[16 + 2048:16 + 2048 + 4]
                # 8 bytes between crc and ecc are reserved
                self.ecc = rawSector[16 + 2048 + 4 + 8:]
            else:
                self.data = rawSector[16:]
                self.crc32 = None
                self.ecc = None
        
gameCD = CDImage(gameChd)
len(gameCD.allSectors)

255896

# Output Format

Designed to be extremely easy to load into memory.

## File Structure

- First, there's a 64-bit offset. This is the offset to a data blob that will be referenced in the JSON metadata.
  The offset is in big-endian order.
- Then a length-prefixed JSON string with all the metadata. The length value is 64-bit unsigned integer in big-endian
  order.
- Next comes at least one null byte. This allows the string to be interpreted as a null-terminated string if needed.
- Then the file is padded with 0's to 8-byte align the data blob.
- Finally, a blob of binary data.

## JSON Structure


```
root object {
    "sectors": array of Sector,
}

Sector object {
    "mode": unsigned integer, // "AUDIO", "MODE1", or "MODE2"
    "dataOffset": unsigned integer, // Offset from the start of the blob to the data.
    "dataLength": unsigned integer, // 2352, 2336, or 2048
    "minute": unsigned integer, // maximum 99 inclusive
    "second": unsigned integer, // maximum 59 inclusive
    "frame": unsigned integer // maximum 75 inclusive
}
```

In [156]:

def imageToFile(image, f):
    sectorArray = []
    blobOffset = 0
    
    
    for i, sector in enumerate(image.allSectors):
        if i % 10000 == 0:
            print("metadata", i, "/", len(image.allSectors))
        metadata = {
            "dataOffset": blobOffset,
            "dataLength": len(sector.data),
            "minute": sector.minutes,
            "second": sector.seconds,
            "frame": sector.sectors,
        }
        blobOffset += len(sector.data)
        if sector.kind == "rawAudio":
            metadata["mode"] = "AUDIO"
        else:
            assert sector.mode > 0 and sector.mode <= 2
            metadata["mode"] = "MODE" + str(sector.mode)
        
        sectorArray.append(metadata)
    
    metadata = json.dumps({"sectors": sectorArray}, separators=(',', ':'))
    padding = 8 - ((len(metadata) + 1) % 8) + 1
    blobStart = 16 + len(metadata) + padding
    
    binaryMetadata = struct.pack("QQ", blobStart, len(metadata))
    binaryMetadata += bytes(metadata, "utf-8")
    binaryMetadata += bytes([0] * padding)
    assert len(binaryMetadata) == blobStart
    
    f.write(binaryMetadata)
    
    for i, sector in enumerate(image.allSectors):
        if i % 10000 == 0:
            print("blob", i, "/", len(image.allSectors))
        f.write(sector.data)
    

with open(filename.replace(".chd", ".dat"), "wb") as f:
    imageToFile(gameCD, f)

print("done")

metadata 0 / 255896
metadata 10000 / 255896
metadata 20000 / 255896
metadata 30000 / 255896
metadata 40000 / 255896
metadata 50000 / 255896
metadata 60000 / 255896
metadata 70000 / 255896
metadata 80000 / 255896
metadata 90000 / 255896
metadata 100000 / 255896
metadata 110000 / 255896
metadata 120000 / 255896
metadata 130000 / 255896
metadata 140000 / 255896
metadata 150000 / 255896
metadata 160000 / 255896
metadata 170000 / 255896
metadata 180000 / 255896
metadata 190000 / 255896
metadata 200000 / 255896
metadata 210000 / 255896
metadata 220000 / 255896
metadata 230000 / 255896
metadata 240000 / 255896
metadata 250000 / 255896
blob 0 / 255896
blob 10000 / 255896
blob 20000 / 255896
blob 30000 / 255896
blob 40000 / 255896
blob 50000 / 255896
blob 60000 / 255896
blob 70000 / 255896
blob 80000 / 255896
blob 90000 / 255896
blob 100000 / 255896
blob 110000 / 255896
blob 120000 / 255896
blob 130000 / 255896
blob 140000 / 255896
blob 150000 / 255896
blob 160000 / 255896
blob 170000 / 255896
